# Summary

Load datasets to S3

In [10]:
import os, sys
import pandas as pd
import json

# AWS Python
import boto3

# Numantic utilities
utils_path = "/Users/stephengodfrey/Documents/Workbench/Numantic/utilities/.."
sys.path.insert(0, utils_path)
from utilities.osa_tools.authentication import ApiAuthentication

api_configs = ApiAuthentication(client="Numantic")



## Read local data


In [2]:
input_data_path = "../data/rag_eval_dataset"
docs_filename = "documents.csv"
multi_pas_qs = "multi_passage_answer_questions.csv"
no_answer_qs = "no_answer_questions.csv"
single_pas_answer_qs = "single_passage_answer_questions.csv"

# Read local data into Pandas dataframes
df_docs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, docs_filename))
df_mpqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, multi_pas_qs))
df_noaqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, no_answer_qs))
df_spqs = pd.read_csv(filepath_or_buffer=os.path.join(input_data_path, single_pas_answer_qs))

## Add some metadata for testing

In [3]:
source_type_map = {"https://enterthegungeon.fandom.com/wiki/Bullet_Kin": "gaming",
                   "https://www.dropbox.com/scl/fi/ljtdg6eaucrbf1aksw5rm/c2%20-%20session%2050%20-%20underground.docx?rlkey=ioqwgkd14i5xk20i3fp38nzgs&e=1&dl=0": "gaming",
                   "https://bytes-and-nibbles.web.app/bytes/stici-note-part-1-planning-and-prototyping": "data_science",
                   "https://github.com/llmware-ai/llmware": "data_science",
                   "https://docs.marimo.io/recipes.html": "recipes",
                   "https://towardsdatascience.com/how-to-maximize-your-impact-as-a-data-scientist-3881995a9cb1": "data_science",
                   "https://ec.europa.eu/commission/presscorner/detail/en/QANDA_21_1683": "government",
                   "https://bg3.wiki/wiki/The_Emperor": "gaming",
                   "https://whattocook.substack.com/p/so-into-northern-spain": "recipes",
                   "https://dmtalkies.com/the-zone-of-interest-ending-explained-and-summary-2023-film/": "entertainment",
                   "https://www.loonyparty.com/about/policy-proposals/": "entertainment",
                   "https://timdettmers.com/2023/01/30/which-gpu-for-deep-learning/": "data_science",
                   "https://gleam.run/cheatsheets/gleam-for-python-users/": "data_science",
                   "https://towardsdatascience.com/gpt-from-scratch-with-mlx-acf2defda30e": "data_science",
                   "https://blog.reedsy.com/short-story/a3gstd/": "entertainment",
                   "http://www.chakoteya.net/DoctorWho/40-1.html": "entertainment",
                   "https://stardewvalleywiki.com/Version_History": "gaming",
                   "https://alanwake.fandom.com/wiki/Alan_Wake_2": "gaming",
                   "https://www.polygon.com/23691206/best-fantasy-books-sci-fi-2023": "entertainment",
                   "https://arxiv.org/pdf/2404.10981": "data_science"
                   }

df_docs["source_type"] = df_docs["source_url"].map(source_type_map)

In [4]:
print("Source documents")
display(df_docs.head())
display(pd.DataFrame(df_docs["source_type"].value_counts()))

print("Single-passage questions")
display(df_spqs.head())


Source documents


,index,source_url,text,source_type
0,0,https://enterthegungeon.fandom.com/wiki/Bullet...,Bullet Kin\nBullet Kin are one of the most com...,gaming
1,1,https://www.dropbox.com/scl/fi/ljtdg6eaucrbf1a...,---The Paths through the Underground/Underdark...,gaming
2,2,https://bytes-and-nibbles.web.app/bytes/stici-...,Semantic and Textual Inference Chatbot Interfa...,data_science
3,3,https://github.com/llmware-ai/llmware,llmware\n\nBuilding Enterprise RAG Pipelines w...,data_science
4,4,https://docs.marimo.io/recipes.html,Recipes\nThis page includes code snippets or “...,recipes


,count
source_type,
data_science,7
gaming,5
entertainment,5
recipes,2
government,1


Single-passage questions


,document_index,question,answer
0,0,What do keybullet kin drop?,Keybullet kin drop a key upon death.
1,0,What kind of gun does the bandana bullet kin use?,The bandana bullet kin wields a machine pistol.
2,1,What do the giants look like?,"One giant is burly, grey-skinned, and 20 feet ..."
3,1,What happens on day 2?,"After a few miles of winding tunnel, you emerg..."
4,2,What were the requirements for the project?,The tool had the following requirements:\n- Ch...


## Load documents to S3

In [11]:
def process_and_upload(df):
    """
    Prepare and load documents to S3. Documents are prepared by (1) adding passage numbers
    associated with double line breaks (/n/n) and (2) creating a metadata sidecar. Text is in the primary document and all other metadata in the sidecar:

    Text Document: doc_99.pdf
    Metadata Sidecar File: doc_99.txt.metadata.json


    :param df:
    :return:
    """
    for idx in df.index:

        ### Step 1: Get text and metadat fields
        raw_text = df.loc[idx, 'text']
        source_url = df.loc[idx, 'source_url']
        source_type = df.loc[idx, 'source_type']

        ### Step 2: Annotate Passage Splits ---
        # Split by double newline and prepend a passage marker
        blocks = raw_text.split('\n\n')
        annotated_text = ""
        for i, block in enumerate(blocks, 1):
            annotated_text += f"[Passage #{i}]\n{block}\n\n"

        ### Step 3: Create Metadata
        metadata = {
            "metadataAttributes": {
                "source_url": source_url,
                "source_type": source_type,
                "doc_index": int(idx),
            }
        }

        ### Step 4: Create filenames
        file_name = f"doc_{idx}.txt"
        metadata_name = f"{file_name}.metadata.json"

        ### Step 5: Upload Text File
        s3_client.put_object(
            Bucket=bucket,
            Key=file_name,
            Body=annotated_text.encode('utf-8')
        )

        ### Step 6: Metadata Sidecar
        s3_client.put_object(
            Bucket=bucket,
            Key=metadata_name,
            Body=json.dumps(metadata).encode('utf-8')
        )

    print(f"Successfully uploaded {len(df)} documents and metadata files.")



In [12]:
### Step 1. Set up session
region_name = 'us-east-2'
session = boto3.Session(profile_name='ns-admin')

### Step 2. Export documents to S3 in Bedrock-friendly format
s3_client = session.client('s3')
bucket = 'ashoka-search-tests'

### Step 3. Delete existing documents first
print("Deleting existing documents...")
paginator = s3_client.get_paginator('list_objects_v2')
for page in paginator.paginate(Bucket=bucket, Prefix='documents/'):
    if 'Contents' in page:
        for obj in page['Contents']:
            s3_client.delete_object(Bucket=bucket, Key=obj['Key'])
print("✓ Existing documents deleted")

### Step 4. Upload text and metadata files
process_and_upload(df=df_docs)


Deleting existing documents...
✓ Existing documents deleted
Successfully uploaded 20 documents and metadata files.


## Create a Bedrock Knowlege Base

In [7]:
"""
Complete Guide to Creating an Amazon Bedrock Knowledge Base
============================================================

Prerequisites:
1. S3 bucket with .txt files containing document text
2. AWS credentials configured
3. Python packages: boto3, opensearchpy

This script assumes .txt files are already uploaded to S3
"""

import boto3
import json
import time

# ============================================================================
# STEP 0: Configuration - Update these values for your use case
# ============================================================================

REGION_NAME = 'us-east-2'  # AWS region
BUCKET_NAME = 'ashoka-search-tests'  # S3 bucket containing documents
S3_PREFIX = 'documents/'  # S3 prefix where .txt files are stored
COLLECTION_NAME = 'ashoka-kb-collection'  # OpenSearch Serverless collection name
KB_NAME = 'ashoka-search-kb'  # Knowledge Base name
INDEX_NAME = 'bedrock-knowledge-base-default-index'  # OpenSearch index name
ROLE_NAME = 'AmazonBedrockExecutionRoleForKnowledgeBase'  # IAM role name
EMBEDDING_MODEL = 'amazon.titan-embed-text-v2:0'  # Bedrock embedding model

# ============================================================================
# STEP 1: Initialize AWS clients and get account information
# ============================================================================

session = boto3.Session()
sts_client = session.client('sts')
account_id = sts_client.get_caller_identity()['Account']

aoss_client = session.client('opensearchserverless', region_name=REGION_NAME)
iam_client = session.client('iam')
bedrock_agent = session.client('bedrock-agent', region_name=REGION_NAME)

print(f"AWS Account ID: {account_id}")
print(f"Region: {REGION_NAME}")

# ============================================================================
# STEP 2: Create OpenSearch Serverless Security Policies
# ============================================================================

# Step 2a: Create encryption policy
encryption_policy = {
    "Rules": [
        {
            "ResourceType": "collection",
            "Resource": [f"collection/{COLLECTION_NAME}"]
        }
    ],
    "AWSOwnedKey": True
}

try:
    aoss_client.create_security_policy(
        name=f'{COLLECTION_NAME}-encryption',
        type='encryption',
        policy=json.dumps(encryption_policy)
    )
    print("✓ Encryption policy created")
except aoss_client.exceptions.ConflictException:
    print("✓ Encryption policy already exists")

# Step 2b: Create network policy
network_policy = [
    {
        "Rules": [
            {
                "ResourceType": "collection",
                "Resource": [f"collection/{COLLECTION_NAME}"]
            },
            {
                "ResourceType": "dashboard",
                "Resource": [f"collection/{COLLECTION_NAME}"]
            }
        ],
        "AllowFromPublic": True
    }
]

try:
    aoss_client.create_security_policy(
        name=f'{COLLECTION_NAME}-network',
        type='network',
        policy=json.dumps(network_policy)
    )
    print("✓ Network policy created")
except aoss_client.exceptions.ConflictException:
    print("✓ Network policy already exists")

# Step 2c: Create data access policy
current_user_arn = sts_client.get_caller_identity()['Arn']
print(f"Current user ARN: {current_user_arn}")

data_access_policy = [
    {
        "Rules": [
            {
                "ResourceType": "collection",
                "Resource": [f"collection/{COLLECTION_NAME}"],
                "Permission": [
                    "aoss:CreateCollectionItems",
                    "aoss:DeleteCollectionItems",
                    "aoss:UpdateCollectionItems",
                    "aoss:DescribeCollectionItems"
                ]
            },
            {
                "ResourceType": "index",
                "Resource": [f"index/{COLLECTION_NAME}/*"],
                "Permission": [
                    "aoss:CreateIndex",
                    "aoss:DeleteIndex",
                    "aoss:UpdateIndex",
                    "aoss:DescribeIndex",
                    "aoss:ReadDocument",
                    "aoss:WriteDocument"
                ]
            }
        ],
        "Principal": [
            f"arn:aws:iam::{account_id}:role/{ROLE_NAME}",
            current_user_arn
        ]
    }
]

try:
    aoss_client.create_access_policy(
        name=f'{COLLECTION_NAME}-access',
        type='data',
        policy=json.dumps(data_access_policy)
    )
    print("✓ Data access policy created")
except aoss_client.exceptions.ConflictException:
    existing_policy = aoss_client.get_access_policy(
        name=f'{COLLECTION_NAME}-access',
        type='data'
    )
    existing_policy_doc = existing_policy['accessPolicyDetail']['policy']
    existing_principals = existing_policy_doc[0]['Principal']

    if current_user_arn not in existing_principals:
        print(f"  Updating policy to include current user...")
        existing_policy_doc[0]['Principal'].append(current_user_arn)
        aoss_client.update_access_policy(
            name=f'{COLLECTION_NAME}-access',
            type='data',
            policyVersion=existing_policy['accessPolicyDetail']['policyVersion'],
            policy=json.dumps(existing_policy_doc)
        )
        print("✓ Data access policy updated")
    else:
        print("✓ Data access policy already exists with current user")

# ============================================================================
# STEP 3: Create OpenSearch Serverless Collection
# ============================================================================

try:
    response = aoss_client.create_collection(
        name=COLLECTION_NAME,
        type='VECTORSEARCH'
    )
    collection_arn = response['createCollectionDetail']['arn']
    print(f"✓ Collection created: {collection_arn}")

    print("  Waiting for collection to become active...")
    while True:
        response = aoss_client.batch_get_collection(names=[COLLECTION_NAME])
        status = response['collectionDetails'][0]['status']
        if status == 'ACTIVE':
            collection_endpoint = response['collectionDetails'][0]['collectionEndpoint']
            print(f"  Collection is active. Endpoint: {collection_endpoint}")
            break
        time.sleep(10)

except aoss_client.exceptions.ConflictException:
    response = aoss_client.batch_get_collection(names=[COLLECTION_NAME])
    collection_arn = response['collectionDetails'][0]['arn']
    collection_endpoint = response['collectionDetails'][0]['collectionEndpoint']
    print(f"✓ Collection already exists: {collection_arn}")

# ============================================================================
# STEP 4: Create IAM Role for Bedrock
# ============================================================================

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "bedrock.amazonaws.com"
            },
            "Action": "sts:AssumeRole"
        }
    ]
}

role_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "s3:GetObject",
                "s3:ListBucket"
            ],
            "Resource": [
                f"arn:aws:s3:::{BUCKET_NAME}",
                f"arn:aws:s3:::{BUCKET_NAME}/*"
            ]
        },
        {
            "Effect": "Allow",
            "Action": [
                "aoss:APIAccessAll"
            ],
            "Resource": [
                collection_arn
            ]
        },
        {
            "Effect": "Allow",
            "Action": [
                "bedrock:InvokeModel"
            ],
            "Resource": [
                f"arn:aws:bedrock:{REGION_NAME}::foundation-model/{EMBEDDING_MODEL}"
            ]
        }
    ]
}

try:
    role_response = iam_client.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description='Role for Bedrock Knowledge Base to access S3 and OpenSearch'
    )
    role_arn = role_response['Role']['Arn']
    print(f"✓ IAM role created: {role_arn}")

    iam_client.put_role_policy(
        RoleName=ROLE_NAME,
        PolicyName='BedrockKnowledgeBasePolicy',
        PolicyDocument=json.dumps(role_policy)
    )
    print("✓ IAM policy attached to role")

    print("  Waiting for IAM role to propagate...")
    time.sleep(10)

except iam_client.exceptions.EntityAlreadyExistsException:
    role_arn = f"arn:aws:iam::{account_id}:role/{ROLE_NAME}"
    print(f"✓ IAM role already exists: {role_arn}")

# ============================================================================
# STEP 5: Delete Existing Knowledge Base (if any)
# NOTE: We skip manual index creation - Bedrock will create it automatically
# ============================================================================

try:
    kbs = bedrock_agent.list_knowledge_bases()
    for kb in kbs['knowledgeBaseSummaries']:
        if kb['name'] == KB_NAME:
            existing_kb_id = kb['knowledgeBaseId']
            print(f"  Found existing Knowledge Base: {existing_kb_id}")
            bedrock_agent.delete_knowledge_base(knowledgeBaseId=existing_kb_id)
            print(f"✓ Deleted existing Knowledge Base")
            print("  Waiting for deletion to complete...")
            time.sleep(15)  # Wait longer for full cleanup
            break
except Exception as e:
    print(f"  No existing Knowledge Base to delete")

# ============================================================================
# STEP 6: Create Knowledge Base (Bedrock will create the index automatically)
# ============================================================================

print("Creating new Knowledge Base...")
response = bedrock_agent.create_knowledge_base(
    name=KB_NAME,
    roleArn=role_arn,
    knowledgeBaseConfiguration={
        'type': 'VECTOR',
        'vectorKnowledgeBaseConfiguration': {
            'embeddingModelArn': f'arn:aws:bedrock:{REGION_NAME}::foundation-model/{EMBEDDING_MODEL}'
        }
    },
    storageConfiguration={
        'type': 'OPENSEARCH_SERVERLESS',
        'opensearchServerlessConfiguration': {
            'collectionArn': collection_arn,
            'vectorIndexName': INDEX_NAME,
            'fieldMapping': {
                'vectorField': 'bedrock-knowledge-base-default-vector',
                'textField': 'AMAZON_BEDROCK_TEXT_CHUNK',
                'metadataField': 'AMAZON_BEDROCK_METADATA'
            }
        }
    }
)

kb_id = response['knowledgeBase']['knowledgeBaseId']
print(f"✓ Knowledge Base created")
print(f"  Knowledge Base ID: {kb_id}")
print("  Waiting for Knowledge Base to initialize...")
time.sleep(5)  # Give time for index creation

# ============================================================================
# STEP 7: Add S3 Data Source to Knowledge Base
# ============================================================================

ds_response = bedrock_agent.create_data_source(
    knowledgeBaseId=kb_id,
    name='s3-txt-docs',
    dataSourceConfiguration={
        'type': 'S3',
        's3Configuration': {
            'bucketArn': f'arn:aws:s3:::{BUCKET_NAME}',
            'inclusionPrefixes': [S3_PREFIX]
        }
    }
)

data_source_id = ds_response['dataSource']['dataSourceId']
print(f"✓ Data source added")
print(f"  Data Source ID: {data_source_id}")

# ============================================================================
# STEP 8: Start Ingestion Job
# ============================================================================

ingestion_response = bedrock_agent.start_ingestion_job(
    knowledgeBaseId=kb_id,
    dataSourceId=data_source_id
)

ingestion_job_id = ingestion_response['ingestionJob']['ingestionJobId']
print(f"✓ Ingestion job started")
print(f"  Ingestion Job ID: {ingestion_job_id}")

# ============================================================================
# STEP 9: Monitor Ingestion Job Status
# ============================================================================

print("\nMonitoring ingestion job...")
while True:
    job_response = bedrock_agent.get_ingestion_job(
        knowledgeBaseId=kb_id,
        dataSourceId=data_source_id,
        ingestionJobId=ingestion_job_id
    )
    status = job_response['ingestionJob']['status']
    print(f"  Status: {status}")

    if status in ['COMPLETE', 'FAILED']:
        if status == 'COMPLETE':
            stats = job_response['ingestionJob']['statistics']
            print(f"  ✓ Documents scanned: {stats.get('numberOfDocumentsScanned', 0)}")
            print(f"  ✓ Documents indexed: {stats.get('numberOfNewDocumentsIndexed', 0)}")
        else:
            print(f"  ✗ Ingestion failed")
            if 'failureReasons' in job_response['ingestionJob']:
                print("  Failure reasons:")
                for reason in job_response['ingestionJob']['failureReasons']:
                    print(f"    - {reason}")
        break

    time.sleep(10)

# ============================================================================
# Setup Complete! Summary
# ============================================================================

print("\n" + "=" * 70)
print("SETUP COMPLETE!")
print("=" * 70)
print(f"Knowledge Base ID: {kb_id}")
print(f"Data Source ID: {data_source_id}")
print(f"Collection ARN: {collection_arn}")
print(f"Collection Endpoint: {collection_endpoint}")
print(f"IAM Role ARN: {role_arn}")
print("\nYou can now query your knowledge base using the Bedrock API:")
print(f"  bedrock_agent_runtime.retrieve(knowledgeBaseId='{kb_id}', ...)")
print("=" * 70)


AWS Account ID: 584560776394
Region: us-east-2
✓ Encryption policy already exists
✓ Network policy already exists
Current user ARN: arn:aws:sts::584560776394:assumed-role/AWSReservedSSO_AdministratorAccess_945db37afb615ee4/Numantic-SGodfrey
✓ Data access policy already exists with current user
✓ Collection already exists: arn:aws:aoss:us-east-2:584560776394:collection/v7mmbvbyqwr3by4j43a6
✓ IAM role already exists: arn:aws:iam::584560776394:role/AmazonBedrockExecutionRoleForKnowledgeBase
  Found existing Knowledge Base: JWPOM15O9H
✓ Deleted existing Knowledge Base
  Waiting for deletion to complete...
Creating new Knowledge Base...
✓ Knowledge Base created
  Knowledge Base ID: GWA4Q9Y2BG
  Waiting for Knowledge Base to initialize...
✓ Data source added
  Data Source ID: ENYJB5RNCU
✓ Ingestion job started
  Ingestion Job ID: TX21EOG54L

Monitoring ingestion job...
  Status: STARTING
  Status: IN_PROGRESS
  Status: IN_PROGRESS
  Status: COMPLETE
  ✓ Documents scanned: 20
  ✓ Documents in

## Check status of ingestion job

In [8]:
# Check ingestion job failure details
failure_response = bedrock_agent.get_ingestion_job(
    knowledgeBaseId=kb_id,
    dataSourceId=data_source_id,
    ingestionJobId=ingestion_job_id
)

print("Ingestion Job Failure Details:")
print(f"Status: {failure_response['ingestionJob']['status']}")

if 'failureReasons' in failure_response['ingestionJob']:
    print("\nFailure Reasons:")
    for reason in failure_response['ingestionJob']['failureReasons']:
        print(f"  - {reason}")

# Print full job details for debugging
print("\nFull Job Details:")
import json

print(json.dumps(failure_response['ingestionJob'], indent=2, default=str))


Ingestion Job Failure Details:
Status: COMPLETE

Full Job Details:
{
  "knowledgeBaseId": "GWA4Q9Y2BG",
  "dataSourceId": "ENYJB5RNCU",
  "ingestionJobId": "TX21EOG54L",
  "status": "COMPLETE",
  "statistics": {
    "numberOfDocumentsScanned": 20,
    "numberOfMetadataDocumentsScanned": 0,
    "numberOfNewDocumentsIndexed": 20,
    "numberOfModifiedDocumentsIndexed": 0,
    "numberOfMetadataDocumentsModified": 0,
    "numberOfDocumentsDeleted": 0,
    "numberOfDocumentsFailed": 0
  },
  "startedAt": "2026-03-31 22:34:36.171278+00:00",
  "updatedAt": "2026-03-31 22:35:00.581473+00:00"
}


## Test search the knowledge base

need to apply BedrockInferenceProfilePolicy to user

In [14]:
# # Initialize Bedrock Agent Runtime client for querying
# region_name = 'us-east-2'
# kb_id = "GWA4Q9Y2BG"
# session = boto3.Session()
# bedrock_runtime = session.client('bedrock-agent-runtime', region_name=region_name)
#
# # Query the knowledge base
# query = "What do keybullet kin drop?"
#
# response = bedrock_runtime.retrieve(
#     knowledgeBaseId=kb_id,
#     retrievalQuery={
#         'text': query
#     },
#     retrievalConfiguration={
#         'vectorSearchConfiguration': {
#             'numberOfResults': 5
#         }
#     }
# )
#
# print(f"Query: {query}\n")
# print("=" * 70)
# print("RETRIEVAL RESULTS:")
# print("=" * 70)
#
# for i, result in enumerate(response['retrievalResults'], 1):
#     print(f"\n--- Result {i} (Score: {result['score']:.4f}) ---")
#     print(f"Content: {result['content']['text'][:500]}...")
#
#     if 'location' in result:
#         if 's3Location' in result['location']:
#             print(f"Source: {result['location']['s3Location']['uri']}")
#
# print("\n" + "=" * 70)
#
# # Use retrieve_and_generate with Claude 3 Sonnet (a stable, available model)
# print("\nGENERATING ANSWER WITH RAG...")
# print("=" * 70)
#
# # Constructing the ARN from the Model ID
# selected_model_id = 'amazon.nova-lite-v1:0'
# model_arn = f'arn:aws:bedrock:{region_name}::foundation-model/{selected_model_id}'
#
#
# account_id = "584560776394"
# # model_arn = f'arn:aws:bedrock:{region_name}:{account_id}:inference-profile/amazon.nova-lite-v1:0'
# model_arn = f'arn:aws:bedrock:us-east-2:{account_id}:inference-profile/us.amazon.nova-lite-v1:0'
# print(model_arn)
#
# rag_response = bedrock_runtime.retrieve_and_generate(
#     input={'text': query},
#     retrieveAndGenerateConfiguration={
#         'type': 'KNOWLEDGE_BASE',
#         'knowledgeBaseConfiguration': {
#             'knowledgeBaseId': kb_id,
#             'modelArn': model_arn
#         }
#     }
# )
#
# print(f"\nAnswer: {rag_response['output']['text']}")
# print(f"\nSources used:")
# for citation in rag_response['citations']:
#     for reference in citation['retrievedReferences']:
#         if 'location' in reference:
#             if 's3Location' in reference['location']:
#                 print(f"  - {reference['location']['s3Location']['uri']}")
#
# print("=" * 70)

Query: What do keybullet kin drop?

RETRIEVAL RESULTS:

--- Result 1 (Score: 0.6843) ---
Content: However, if the player does not manage to kill them in time, they will disappear.  Unlike other Bullet Kin, Keybullet Kin do not deal contact damage if they run into the player.  Jammed Keybullet Kin drop 2 keys instead of 1. These Jammed variations run faster and will take less time to teleport away from the player if they are not destroyed quickly.  If a Keybullet Kin is knocked into a pit, it will not drop a key.  The chances for a specific number of Keybullet Kin to spawn on a floor are:  0	...
Source: s3://ashoka-search-tests/documents/doc_0.txt

--- Result 2 (Score: 0.5349) ---
Content: Chance Kin Chance Kin run away from the player, and drop a random pickup upon death. However, if the player does not manage to kill them in time, they will disappear. Jammed Chance Kins have a chance to drop twice the loot.  The chances for a specific number of Chance Kin to spawn on a floor are:  0	1

    ## Check which models are available

In [7]:
region_name

'us-east-2'

In [8]:
bedrock = boto3.client(service_name='bedrock', region_name=region_name) # Change to your region
response = bedrock.list_foundation_models()

for model in response['modelSummaries']:
    print(model['modelId'])

nvidia.nemotron-nano-12b-v2
anthropic.claude-sonnet-4-20250514-v1:0
anthropic.claude-haiku-4-5-20251001-v1:0
qwen.qwen3-235b-a22b-2507-v1:0
moonshotai.kimi-k2.5
openai.gpt-oss-120b-1:0
stability.stable-creative-upscale-v1:0
qwen.qwen3-next-80b-a3b
deepseek.v3.2
nvidia.nemotron-nano-3-30b
anthropic.claude-sonnet-4-6
minimax.minimax-m2
zai.glm-4.7-flash
mistral.voxtral-mini-3b-2507
amazon.nova-pro-v1:0
stability.stable-image-remove-background-v1:0
stability.stable-image-control-sketch-v1:0
amazon.nova-2-lite-v1:0
stability.stable-conservative-upscale-v1:0
minimax.minimax-m2.5
google.gemma-3-12b-it
stability.stable-image-search-recolor-v1:0
moonshot.kimi-k2-thinking
mistral.mistral-large-3-675b-instruct
twelvelabs.pegasus-1-2-v1:0
mistral.devstral-2-123b
minimax.minimax-m2.1
nvidia.nemotron-super-3-120b
qwen.qwen3-32b-v1:0
mistral.ministral-3-14b-instruct
anthropic.claude-opus-4-6-v1
writer.palmyra-x5-v1:0
nvidia.nemotron-nano-9b-v2
mistral.ministral-3-8b-instruct
mistral.voxtral-small-24